# Data Collection Pipeline

This notebook fetches climatology data from the **NASA POWER API**, extracts solar **PVOUT** values from a raster (`.tif`) file, merges both datasets by location, and exports the results to CSV.

**Pipeline steps:**
1. Setup
2. Configuration
3. Climatology Data Fetching
4. Climatology Data Processing
5. Solar Data Extraction
6. Merge Datasets
7. Export Final Dataset
8. Cleaned up root folder

---
## 1. Setup

Add the local `lib` directory to the Python path so that installed dependencies can be imported.

In [1]:
import sys
sys.path.insert(0, './lib')

In [2]:
# Importing dependencies

# dependencies for os level tasks ("sys" is already imported, hence commented out)
import os, shutil #, sys

# dependencies for access NASA POWER API
import time, json, urllib3, requests #, sys
from concurrent.futures import ThreadPoolExecutor

# dependencies for data processing
import pandas as pd, glob, json

# dependencies for GIS Raster data extraction
import rasterio

---
## 2. Configuration

Set the raster file path, download/export options, NASA POWER API request template, and target locations.

In [3]:
#=================================================================================================
# 1. Select download Options:
#=================================================================================================

download_option = 3               # 1 = Final Dataset Only, 2 = Raw Datasets only, 3 = Everything.


#=================================================================================================
# 2. Set NASA POWER API parameters
#=================================================================================================

# Instructions:
# This is where you can tweak NASA POWER API request.
# URL in "parameters_to_request" determines WHAT data you need to request.

# Monthly API uses YYYY format for start/end or YYYYMM.
parameters_to_request = r"https://power.larc.nasa.gov/api/temporal/monthly/point?parameters=T2M,ALLSKY_SFC_SW_DWN,RH2M,WS2M,CLOUD_AMT,PRECTOTCORR&community=RE&longitude={longitude}&latitude={latitude}&start=1999&end=2018&format=JSON"


#=================================================================================================
# 3. Set locations
#=================================================================================================

# "locations" list determines for WHERE exactly you need that data.
# Use (latitude, longitude) format.

locations = [
    (7.297, 81.679),  # Ampara, Ampara
    (7.416, 81.816),  # Ampara, Kalmunai
    (7.371, 81.802),  # Ampara, Samanthurai
    (7.218, 81.849),  # Ampara, Akkaraipattu
    (6.878, 81.826),  # Ampara, Pottuvil
    (8.335, 80.410),  # Anuradhapura, Anuradhapura
    (8.041, 80.590),  # Anuradhapura, Kekirawa
    (8.536, 80.485),  # Anuradhapura, Medawachchiya
    (8.152, 80.407),  # Anuradhapura, Eppawala
    (8.151, 80.294),  # Anuradhapura, Tambuttegama
    (6.989, 81.057),  # Badulla, Badulla
    (6.830, 80.990),  # Badulla, Bandarawela
    (6.768, 80.957),  # Badulla, Haputale
    (6.903, 80.900),  # Badulla, Welimada
    (7.320, 80.985),  # Badulla, Mahiyanganaya
    (7.735, 81.694),  # Batticaloa, Batticaloa
    (7.691, 81.725),  # Batticaloa, Kattankudy
    (7.773, 81.604),  # Batticaloa, Eravur
    (7.917, 81.545),  # Batticaloa, Valachchenai
    (7.545, 81.789),  # Batticaloa, Kaluwanchikudy
    (6.926, 79.860),  # Colombo, Colombo
    (6.839, 79.866),  # Colombo, Dehiwala-Mount Lavinia
    (6.773, 79.881),  # Colombo, Moratuwa
    (6.888, 79.918),  # Colombo, Kotte
    (6.848, 79.927),  # Colombo, Maharagama
    (6.032, 80.215),  # Galle, Galle
    (6.234, 80.052),  # Galle, Ambalangoda
    (6.265, 80.141),  # Galle, Elpitiya
    (6.138, 80.106),  # Galle, Hikkaduwa
    (6.064, 80.228),  # Galle, Karapitiya
    (7.091, 79.999),  # Gampaha, Gampaha
    (7.208, 79.835),  # Gampaha, Negombo
    (6.953, 79.919),  # Gampaha, Kelaniya
    (6.985, 79.891),  # Gampaha, Wattala
    (7.168, 79.948),  # Gampaha, Minuwangoda
    (6.124, 81.125),  # Hambantota, Hambantota
    (6.024, 80.796),  # Hambantota, Tangalle
    (6.279, 81.288),  # Hambantota, Tissamaharama
    (6.124, 81.026),  # Hambantota, Ambalantota
    (6.046, 80.757),  # Hambantota, Beliatta
    (9.664, 80.016),  # Jaffna, Jaffna
    (9.816, 80.233),  # Jaffna, Point Pedro
    (9.658, 80.158),  # Jaffna, Chavakachcheri
    (9.673, 80.026),  # Jaffna, Nallur
    (9.813, 80.040),  # Jaffna, Kankesanthurai
    (6.586, 79.960),  # Kalutara, Kalutara
    (6.713, 79.907),  # Kalutara, Panadura
    (6.717, 80.063),  # Kalutara, Horana
    (6.478, 79.983),  # Kalutara, Beruwala
    (6.522, 80.114),  # Kalutara, Matugama
    (7.293, 80.635),  # Kandy, Kandy
    (7.161, 80.573),  # Kandy, Gampola
    (7.268, 80.593),  # Kandy, Peradeniya
    (7.054, 80.534),  # Kandy, Nawalapitiya
    (7.323, 80.621),  # Kandy, Katugastota
    (7.253, 80.345),  # Kegalle, Kegalle
    (7.254, 80.446),  # Kegalle, Mawanella
    (7.325, 80.395),  # Kegalle, Rambukkana
    (7.042, 80.252),  # Kegalle, Ruwanwella
    (7.224, 80.197),  # Kegalle, Warakapola
    (9.383, 80.408),  # Kilinochchi, Kilinochchi
    (9.531, 80.345),  # Kilinochchi, Pallai
    (9.489, 80.198),  # Kilinochchi, Pooneryn
    (9.431, 80.407),  # Kilinochchi, Paranthan
    (9.283, 80.334),  # Kilinochchi, Akkarayankulam
    (7.483, 80.366),  # Kurunegala, Kurunegala
    (7.468, 80.038),  # Kurunegala, Kuliyapitiya
    (7.333, 79.966),  # Kurunegala, Pannala
    (7.426, 80.214),  # Kurunegala, Narammala
    (7.618, 80.222),  # Kurunegala, Wariyapola
    (8.977, 79.913),  # Mannar, Mannar
    (9.079, 79.824),  # Mannar, Pesalai
    (8.835, 80.019),  # Mannar, Murunkan
    (8.847, 79.966),  # Mannar, Nanaddan
    (8.761, 79.882),  # Mannar, Silavathurai
    (7.466, 80.616),  # Matale, Matale
    (7.859, 80.655),  # Matale, Dambulla
    (7.766, 80.569),  # Matale, Galewela
    (7.525, 80.676),  # Matale, Rattota
    (7.419, 80.619),  # Matale, Ukuwela
    (5.950, 80.533),  # Matara, Matara
    (5.973, 80.428),  # Matara, Weligama
    (6.101, 80.481),  # Matara, Akuressa
    (5.968, 80.695),  # Matara, Dikwella
    (6.136, 80.648),  # Matara, Hakmana
    (6.871, 81.348),  # Moneragala, Moneragala
    (6.734, 81.103),  # Moneragala, Wellawaya
    (7.161, 81.222),  # Moneragala, Bibile
    (6.757, 81.238),  # Moneragala, Buttala
    (6.414, 81.332),  # Moneragala, Kataragama
    (9.283, 80.800),  # Mullaitivu, Mullaitivu
    (9.308, 80.665),  # Mullaitivu, Puthukkudiyiruppu
    (9.129, 80.445),  # Mullaitivu, Mankulam
    (9.146, 80.647),  # Mullaitivu, Oddusuddan
    (9.144, 80.285),  # Mullaitivu, Mallavi
    (6.966, 80.766),  # Nuwara Eliya, Nuwara Eliya
    (6.883, 80.596),  # Nuwara Eliya, Hatton
    (6.936, 80.656),  # Nuwara Eliya, Talawakele
    (6.932, 80.741),  # Nuwara Eliya, Nanu Oya
    (7.037, 80.875),  # Nuwara Eliya, Ragala
    (7.940, 81.000),  # Polonnaruwa, Polonnaruwa
    (7.948, 81.026),  # Polonnaruwa, Kaduruwela
    (8.053, 80.980),  # Polonnaruwa, Hingurakgoda
    (8.147, 80.986),  # Polonnaruwa, Medirigiriya
    (7.954, 81.248),  # Polonnaruwa, Welikanda
    (8.034, 79.835),  # Puttalam, Puttalam
    (7.575, 79.799),  # Puttalam, Chilaw
    (7.323, 79.824),  # Puttalam, Wennappuwa
    (7.419, 79.866),  # Puttalam, Nattandiya
    (7.300, 79.882),  # Puttalam, Dankotuwa
    (6.666, 80.400),  # Ratnapura, Ratnapura
    (6.649, 80.706),  # Ratnapura, Balangoda
    (6.619, 80.536),  # Ratnapura, Pelmadulla
    (6.340, 80.841),  # Ratnapura, Embilipitiya
    (6.852, 80.264),  # Ratnapura, Eheliyagoda
    (8.566, 81.233),  # Trincomalee, Trincomalee
    (8.514, 81.183),  # Trincomalee, Kinniya
    (8.455, 81.265),  # Trincomalee, Mutur
    (8.369, 80.999),  # Trincomalee, Kantalai
    (8.809, 81.087),  # Trincomalee, Kuchchaveli
    (8.750, 80.483),  # Vavuniya, Vavuniya
    (8.666, 80.316),  # Vavuniya, Cheddikulam
    (8.905, 80.528),  # Vavuniya, Puliyankulam
    (8.956, 80.771),  # Vavuniya, Nedunkeni
    (8.877, 80.518)   # Vavuniya, Omanthai
]

---
## 3. Climatology Data Fetching

Query the NASA POWER API in parallel for every location defined above and save each response as a JSON file.

In [4]:
# import sys, time, json, urllib3, requests
# from concurrent.futures import ThreadPoolExecutor

urllib3.disable_warnings()

def download_function(collection):
    request, filepath = collection
    response = requests.get(url=request, verify=False, timeout=30.00).json()
    with open(filepath, 'w') as file_object:
        json.dump(response, file_object)

class Process():
    def __init__(self):
        self.threads = 5
        self.request_template = parameters_to_request
        self.filename_template = "File_Lat_{latitude}_Lon_{longitude}.json"
        self.times = {}

    def execute(self):
        Start_Time = time.time()

        requests_list = []
        for latitude, longitude in locations:
            request = self.request_template.format(latitude=latitude, longitude=longitude)
            filename = self.filename_template.format(latitude=latitude, longitude=longitude)
            requests_list.append((request, filename))

        requests_total = len(requests_list)
        with ThreadPoolExecutor(max_workers=self.threads) as executor:
            for i, _ in enumerate(executor.map(download_function, requests_list), 1):
                sys.stderr.write('\rExporting {0:%}'.format(i/requests_total))

        self.times["Total Script"] = round((time.time() - Start_Time), 2)
        print("\nTotal Script Time:", self.times["Total Script"])
        print(f"\nClimatology data fetched successfully!")

Process().execute()

Exporting 99.200000%


Total Script Time: 76.97

Climatology data fetched successfully!


Exporting 100.000000%

---
## 4. Climatology Data Processing

Load the downloaded JSON files and combine them into a single DataFrame.

NASA POWER API fetch data as JSON.

This code segment create a dataframe from fetched data.

In [5]:
# import pandas as pd, glob, json

# Find all generated files
files = glob.glob('File_Lat_*.json')

all_data = []

for file in files:
    with open(file, 'r') as f:
        data = json.load(f)

        if 'properties' in data:
            # Extract parameters
            df_temp = pd.DataFrame(data['properties']['parameter'])

            # Add metadata
            df_temp['latitude'] = data['geometry']['coordinates'][1]
            df_temp['longitude'] = data['geometry']['coordinates'][0]

            df_temp.index.name = 'date'
            df_temp = df_temp.reset_index()
            all_data.append(df_temp)
        else:
            print(f"Warning: Could not process {file}. API Error: {data.get('messages', 'Unknown error')}")

if all_data:
    df_climatology = pd.concat(all_data, ignore_index=True)
#    display(df_climatology.head())
    print(f"\nClimatology data extracted successfully!")
    print(f"\nTotal rows loaded: {len(df_climatology)}")
else:
    print("No data was successfully loaded. Please re-run the ""Climatology data fetching"" first.")


Climatology data extracted successfully!

Total rows loaded: 32500


---
## 5. Solar Data Extraction

Read PVOUT values from the raster file for each location.

In [7]:
# import rasterio
# import pandas as pd

file_path = r"./data/PVOUT.tif"

results = []
with rasterio.open(file_path) as src:
    band = src.read(1)
    for latitude, longitude in locations:
        row, col = src.index(longitude, latitude)
        value = float(band[row, col])
        results.append({
            "latitude": latitude,
            "longitude": longitude,
            "pvout": value
        })

df_solar = pd.DataFrame(results)

# display(df_solar.head())
print(f"\nSolar data extracted successfully!")
print(f"\nTotal rows loaded: {len(df_solar)}")


Solar data extracted successfully!

Total rows loaded: 125


---
## 6. Merge Datasets

Aggregate climatology data by location and merge it with the solar PVOUT dataset.

In [8]:
# clean climatology dataset
df_climatology_2 = df_climatology[df_climatology["date"].str.endswith("13")]
df_climatology_2 = df_climatology_2.groupby(['latitude', 'longitude'])[["ALLSKY_SFC_SW_DWN", "CLOUD_AMT", "PRECTOTCORR", "RH2M", "T2M", "WS2M"]].mean()

# creating a new dataset by merging climatology dataset with solar dataset,
df_final = pd.merge(
    df_climatology_2,
    df_solar,
    left_on=['latitude', 'longitude'],
    right_on=['latitude', 'longitude'],
    how='inner'
)

# display(df_final.head())
# display(df_final.shape)
print(f"\nDataframes merged successfully!")
print(f"\nTotal rows loaded: {len(df_final)}")


Dataframes merged successfully!

Total rows loaded: 125


---
## 7. Export Final Dataset

Save the resulting DataFrame(s) to CSV, based on the `download_option` set in the Configuration section.

In [9]:
# import os

if ('df_final' in globals()) and ((download_option == 1) or (download_option == 3)):
    filename = 'dataset.csv'
    df_final.to_csv(filename, index=False)
    print(f"\n'{filename}': Download Successful!")

if ('df_climatology' in globals()) and ((download_option == 2) or (download_option == 3)):
    filename = 'dataset_climatology.csv'
    df_climatology.to_csv(filename, index=False)
    print(f"\n'{filename}': Download Successful!")

if ('df_solar' in globals()) and ((download_option == 2) or (download_option == 3)):
    filename = 'dataset_solar.csv'
    df_solar.to_csv(filename, index=False)
    print(f"\n'{filename}': Download Successful!")


'dataset.csv': Download Successful!

'dataset_climatology.csv': Download Successful!

'dataset_solar.csv': Download Successful!


---
## 8. Cleaning up root directory

move all .json and .csv files to "data" folder.

In [10]:
# import os, shutil, glob

# Set destination folder
dest_folder = "data"

# Find all JSON and CSV files
files = glob.glob("*.json") + glob.glob("*.csv")

# Move all JSON and CSV files
for file in files:
    shutil.move(file, os.path.join(dest_folder, file))

print(f"\nTotal files moved: {len(files)}")


Total files moved: 128
